# ETL - Vagas escondidas de startups (InHire)

Pipeline estável sem browser.

Funções:

- Descobrir empresas InHire
- Extrair vagas
- Filtrar área de dados
- Filtrar remoto
- Detectar vagas novas
- Gerar dataset histórico


In [ ]:
import pandas as pd
import requests
import re
from datetime import datetime
import os

## Configuração

In [ ]:
DATA_ETL = datetime.now().strftime('%Y-%m-%d')
OUTPUT_FILE = "dataset_vagas_startups.xlsx"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

KEYWORDS_DADOS = [
    "data",
    "analytics",
    "machine learning",
    "ai",
    "cientista",
    "analista",
    "data engineer",
    "bi"
]

## Lista inicial de empresas

Base inicial conhecida de empresas InHire.
Depois o dataset cresce automaticamente.

In [ ]:
EMPRESAS = [
    "sympla",
    "olist",
    "cora",
    "contabilizei",
    "loggi",
    "neon",
    "quintoandar",
    "trybe"
]

## Extrair vagas de uma empresa

In [ ]:
def buscar_vagas_empresa(empresa):

    vagas = []

    try:

        url = f"https://{empresa}.inhire.app/vagas"

        r = requests.get(url, headers=HEADERS)

        html = r.text

        padrao = r'href="(/vagas/.*?)"'

        links = re.findall(padrao, html)

        for link in links:

            full = f"https://{empresa}.inhire.app" + link

            titulo = link.split("/")[-1].replace("-"," ")

            vagas.append({
                "empresa": empresa,
                "vaga": titulo,
                "local": "remote",
                "link": full
            })

    except:
        pass

    return vagas

## Coletar vagas

In [ ]:
def coletar_vagas():

    vagas = []

    for empresa in EMPRESAS:

        print("Escaneando:", empresa)

        v = buscar_vagas_empresa(empresa)

        vagas.extend(v)

    return vagas

## Filtrar vagas de dados

In [ ]:
def filtrar_dados(df):

    pattern = '|'.join(KEYWORDS_DADOS)

    return df[df['vaga'].str.lower().str.contains(pattern, na=False)]

## Detectar vagas novas

In [ ]:
def detectar_novas(df):

    if not os.path.exists(OUTPUT_FILE):

        df['nova_vaga'] = True

        return df

    antigo = pd.read_excel(OUTPUT_FILE)

    antigos_links = set(antigo['link'])

    df['nova_vaga'] = ~df['link'].isin(antigos_links)

    return df

## Pipeline principal

In [ ]:
print("Coletando vagas...")

vagas = coletar_vagas()

df = pd.DataFrame(vagas)

print("Total vagas coletadas:", len(df))

df = filtrar_dados(df)

df['data_etl'] = DATA_ETL

df = detectar_novas(df)

df.to_excel(OUTPUT_FILE, index=False)

print("Dataset salvo:", OUTPUT_FILE)

df.head()